#### How to split JSON data
This json splitter splits json data while allowing control over chunk sizes. It traverses json data depth first and builds smaller json chunks. It attempts to keep nested json objects whole but will split them if needed to keep chunks between a min_chunk_size and the max_chunk_size.

If the value is not a nested json, but rather a very large string the string will not be split. If you need a hard cap on the chunk size consider composing this with a Recursive Text splitter on those chunks. There is an optional pre-processing step to split lists, by first converting them to json (dict) and then splitting them as such.

- How the text is split: json value.
- How the chunk size is measured: by number of characters.


In [5]:
import json
import requests

json_data = requests.get("https://api.smith.langchain.com/openapi.json").json()
print(json.dumps(json_data, indent=4))

{
    "openapi": "3.1.0",
    "info": {
        "title": "LangSmith",
        "description": "The LangSmith API is used to programmatically create and manage LangSmith resources.\n\n## Host\nhttps://api.smith.langchain.com\n\n## Authentication\nTo authenticate with the LangSmith API, set the `X-Api-Key` header\nto a valid [LangSmith API key](https://docs.langchain.com/langsmith/create-account-api-key#create-an-api-key).\n\n",
        "version": "0.1.0"
    },
    "paths": {
        "/api/v1/audit-logs": {
            "get": {
                "tags": [
                    "audit-logs"
                ],
                "summary": "Get Audit Logs",
                "description": "Retrieve audit log records for the authenticated user's organization in OCSF format.\n\nRequires both start_time and end_time parameters to filter logs within a date range.\nSupports cursor-based pagination.\n\nReturns results in OCSF API Activity (Class UID: 6003) format,\nwhich is compatible with security moni

In [8]:
from langchain_text_splitters import RecursiveJsonSplitter

json_splitter = RecursiveJsonSplitter(max_chunk_size=300)
json_chunks = json_splitter.split_json(json_data)
json_chunks[:3]

[{'openapi': '3.1.0',
  'info': {'title': 'LangSmith',
   'description': 'The LangSmith API is used to programmatically create and manage LangSmith resources.\n\n## Host\nhttps://api.smith.langchain.com\n\n## Authentication\nTo authenticate with the LangSmith API, set the `X-Api-Key` header\nto a valid [LangSmith API key](https://docs.langchain.com/langsmith/create-account-api-key#create-an-api-key).\n\n'}},
 {'info': {'version': '0.1.0'},
  'paths': {'/api/v1/audit-logs': {'get': {'tags': ['audit-logs'],
     'summary': 'Get Audit Logs'}}}},
 {'paths': {'/api/v1/audit-logs': {'get': {'description': "Retrieve audit log records for the authenticated user's organization in OCSF format.\n\nRequires both start_time and end_time parameters to filter logs within a date range.\nSupports cursor-based pagination.\n\nReturns results in OCSF API Activity (Class UID: 6003) format,\nwhich is compatible with security monitoring and SIEM tools.\nReference: https://schema.ocsf.io/1.7.0/classes/api_act

In [11]:
## The splitters can also output document

docs = json_splitter.create_documents([json_data])
docs[:3]

[Document(metadata={}, page_content='{"openapi": "3.1.0", "info": {"title": "LangSmith", "description": "The LangSmith API is used to programmatically create and manage LangSmith resources.\\n\\n## Host\\nhttps://api.smith.langchain.com\\n\\n## Authentication\\nTo authenticate with the LangSmith API, set the `X-Api-Key` header\\nto a valid [LangSmith API key](https://docs.langchain.com/langsmith/create-account-api-key#create-an-api-key).\\n\\n"}}'),
 Document(metadata={}, page_content='{"info": {"version": "0.1.0"}, "paths": {"/api/v1/audit-logs": {"get": {"tags": ["audit-logs"], "summary": "Get Audit Logs"}}}}'),
 Document(metadata={}, page_content='{"paths": {"/api/v1/audit-logs": {"get": {"description": "Retrieve audit log records for the authenticated user\'s organization in OCSF format.\\n\\nRequires both start_time and end_time parameters to filter logs within a date range.\\nSupports cursor-based pagination.\\n\\nReturns results in OCSF API Activity (Class UID: 6003) format,\\nw

In [13]:
# In string format

texts = json_splitter.split_text(json_data)
texts[:3]

['{"openapi": "3.1.0", "info": {"title": "LangSmith", "description": "The LangSmith API is used to programmatically create and manage LangSmith resources.\\n\\n## Host\\nhttps://api.smith.langchain.com\\n\\n## Authentication\\nTo authenticate with the LangSmith API, set the `X-Api-Key` header\\nto a valid [LangSmith API key](https://docs.langchain.com/langsmith/create-account-api-key#create-an-api-key).\\n\\n"}}',
 '{"info": {"version": "0.1.0"}, "paths": {"/api/v1/audit-logs": {"get": {"tags": ["audit-logs"], "summary": "Get Audit Logs"}}}}',
 '{"paths": {"/api/v1/audit-logs": {"get": {"description": "Retrieve audit log records for the authenticated user\'s organization in OCSF format.\\n\\nRequires both start_time and end_time parameters to filter logs within a date range.\\nSupports cursor-based pagination.\\n\\nReturns results in OCSF API Activity (Class UID: 6003) format,\\nwhich is compatible with security monitoring and SIEM tools.\\nReference: https://schema.ocsf.io/1.7.0/class